# Capítulo 4: El Bosque y el Boost (Modelos de Árboles)

> *"Un árbol de decisión es como el juego de '20 preguntas' que haces mentalmente."*

**Objetivo:** Predecir la rotación de empleados (Attrition) usando Random Forest, XGBoost y LightGBM, comparar sus rendimientos, y reflexionar sobre los sesgos que los modelos pueden aprender.

## Celda 1: Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, accuracy_score, precision_recall_curve
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    print('XGBoost disponible:', xgb.__version__)
except ImportError:
    print('XGBoost no instalado. Instalar con: pip install xgboost')

try:
    import lightgbm as lgb
    print('LightGBM disponible:', lgb.__version__)
except ImportError:
    print('LightGBM no instalado. Instalar con: pip install lightgbm')

pd.set_option('display.max_columns', 25)
np.random.seed(42)
print('\nLibrerías cargadas correctamente.')

## Celda 2: Carga de datos (HR Analytics)

Dataset de 1500 empleados con datos de RRHH. Variable objetivo: `Attrition` (¿el empleado renunció?).

In [ ]:
df = pd.read_csv('../datos/datos_empleados_hr.csv')

print(f'Dimensiones: {df.shape[0]} filas, {df.shape[1]} columnas')
print(f'\nDistribución de Attrition:')
print(df['Attrition'].value_counts())
print(f'\nTasa de rotación: {df["Attrition"].value_counts(normalize=True)["Yes"]*100:.1f}%')
print(f'\nPrimeras filas:')
df.head(10)

In [ ]:
# Análisis exploratorio rápido
print('=== Info general ===')
print(df.info())
print(f'\n=== Valores nulos ===')
print(df.isnull().sum())
print(f'\n=== Distribución de Gender ===')
print(df['Gender'].value_counts())
print(f'\n=== Ingreso por Género ===')
print(df.groupby('Gender')['MonthlyIncome'].describe().round(0))

## Celda 3: Preprocesamiento

Convertimos variables categóricas a numéricas y separamos features de la variable objetivo.

In [ ]:
# Codificar variables categóricas
le_dict = {}
categorical_cols = ['Department', 'Gender', 'OverTime']

df_processed = df.copy()
for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    le_dict[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Variable objetivo
df_processed['Attrition'] = (df_processed['Attrition'] == 'Yes').astype(int)

# Separar features y target
X = df_processed.drop('Attrition', axis=1)
y = df_processed['Attrition']

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'\nConjunto de entrenamiento: {X_train.shape[0]} muestras')
print(f'Conjunto de prueba: {X_test.shape[0]} muestras')
print(f'\nDistribución en train: {dict(y_train.value_counts())}')
print(f'Distribución en test: {dict(y_test.value_counts())}')
print(f'\nFeatures: {list(X.columns)}')

## Celda 4: Random Forest — "La opinión de 100 expertos"

Random Forest entrena muchos árboles de decisión, cada uno con datos y features diferentes, y promedia sus predicciones.

In [ ]:
# Entrenar Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,       # 200 árboles
    max_depth=15,           # profundidad máxima
    min_samples_split=10,   # mínimo para dividir un nodo
    min_samples_leaf=5,     # mínimo en cada hoja
    max_features='sqrt',    # features por split
    class_weight='balanced',# compensar clases desbalanceadas
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predicciones
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print('=== Random Forest — Resultados ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_rf):.4f}')
print(f'AUC-ROC:  {roc_auc_score(y_test, y_proba_rf):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['No', 'Yes']))

In [ ]:
# Cross-validation
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='f1')
print(f'\nCross-Validation F1: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})')
print(f'Scores por fold: {[f"{s:.4f}" for s in cv_scores]}')

## Celda 5: Feature Importance — "Las preguntas más importantes"

¿Qué features influyen más en la predicción de rotación?

In [ ]:
# Feature Importance del Random Forest
importances = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importancia (Gini)')
plt.tight_layout()
plt.show()

print('\nTop 10 features más importantes:')
for feat, imp in importances.sort_values(ascending=False).head(10).items():
    print(f'  {feat:30s} {imp:.4f}')

### ¿Qué nos dice esto?

Si **OverTime** (horas extras) es la feature #1, ¿es porque el modelo aprendió que los que hacen horas extras se van más? ¿O porque hay un sesgo en los datos?

**Esto es exactamente por qué necesitamos mirar la importancia de features con ojos críticos.**

In [ ]:
# Análisis de las top features vs Attrition
top_features = importances.sort_values(ascending=False).head(5).index.tolist()

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, feat in enumerate(top_features):
    ax = axes[i]
    if df[feat].nunique() <= 10:
        # Categórica: proporción de attrition
        ct = pd.crosstab(df[feat], df['Attrition'], normalize='index')
        ct['Yes'].plot(kind='bar', ax=ax, color='coral', alpha=0.7)
        ax.set_title(f'{feat}\nvs Attrition', fontsize=10)
        ax.set_ylabel('% Attrition')
    else:
        # Numérica: distribución
        df[df['Attrition']=='No'][feat].hist(ax=ax, bins=20, alpha=0.5, label='No', color='steelblue')
        df[df['Attrition']=='Yes'][feat].hist(ax=ax, bins=20, alpha=0.5, label='Yes', color='coral')
        ax.set_title(f'{feat}\nvs Attrition', fontsize=10)
        ax.legend()
plt.tight_layout()
plt.show()

## Celda 6: XGBoost — "Aprender de los errores"

XGBoost entrena árboles en secuencia, donde cada uno corrige los errores del anterior.

In [ ]:
# XGBoost
scale_pos = len(y_train[y_train==0]) / len(y_train[y_train==1])

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    eval_metric='auc',
    random_state=42,
    use_label_encoder=False
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print('=== XGBoost — Resultados ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_xgb):.4f}')
print(f'AUC-ROC:  {roc_auc_score(y_test, y_proba_xgb):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_xgb, target_names=['No', 'Yes']))

## Celda 7: Comparación de modelos

In [ ]:
# LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_train, y_train)

y_pred_lgb = lgb_model.predict(X_test)
y_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]

print('=== LightGBM — Resultados ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lgb):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_lgb):.4f}')
print(f'AUC-ROC:  {roc_auc_score(y_test, y_proba_lgb):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_lgb, target_names=['No', 'Yes']))

In [ ]:
# Tabla comparativa
modelos = {
    'Random Forest': (y_pred_rf, y_proba_rf),
    'XGBoost': (y_pred_xgb, y_proba_xgb),
    'LightGBM': (y_pred_lgb, y_proba_lgb)
}

resultados = []
for nombre, (y_pred, y_proba) in modelos.items():
    resultados.append({
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    })

df_resultados = pd.DataFrame(resultados).set_index('Modelo')
print('=== Comparación de Modelos ===')
print(df_resultados.round(4).to_string())

# Curva ROC
fig, ax = plt.subplots(figsize=(8, 6))
for nombre, (_, y_proba) in modelos.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'{nombre} (AUC={auc:.3f})')
ax.plot([0,1], [0,1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curva ROC — Comparación de Modelos', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## Celda 8: Post-Mortem completo

¿Qué salió bien? ¿Qué salió mal? ¿Qué aprenderíamos para la próxima vez?

In [ ]:
print('='*60)
print('POST-MORTEM — Proyecto de Predicción de Rotación')
print('='*60)

print('\n## ¿Qué salió bien?')
print('- El preprocessing fue limpio: no había nulos ni duplicados.')
print('- Los tres modelos superaron el baseline (accuracy > 0.80).')
print('- Feature Importance reveló patrones útiles (OverTime, MonthlyIncome).')

print('\n## ¿Qué salió mal?')
print('- La clase "Yes" (rotación) está desbalanceada (~14%).')
print('- Sin SMOTE o ajuste de threshold, el modelo favorece la clase mayoritaria.')
print('- No se hizo análisis de features proxy (Gender aparece en importance).')
print('- No hay validación temporal: el split es aleatorio, no cronológico.')

print('\n## Lecciones aprendidas')
print('1. Siempre verificar la distribución de clases ANTES de entrenar.')
print('2. Feature Importance sin contexto ético es peligrosa.')
print('3. Un split aleatorio puede ser unrealista en problemas temporales.')
print('4. Cross-validation es más confiable que un solo split.')

print('\n## ¿Qué haría diferente?')
print('- Usar StratifiedKFold en vez de un solo train/test split.')
print('- Analizar Gender y Department como posibles proxies de sesgo.')
print('- Probar feature engineering: ratios, interacciones.')
print('- Incluir análisis de fairness por subgrupos.')

## Celda 9: Ética — ¿Qué features revelan sesgos?

El modelo aprende del pasado. Si el pasado tiene injusticias, el modelo las replica.

In [ ]:
print('='*60)
print('ANÁLISIS ÉTICO — Sesgos en el Modelo')
print('='*60)

# 1. Brecha salarial por género
print('\n## 1. Brecha salarial por género')
gender_income = df.groupby('Gender')['MonthlyIncome'].agg(['mean', 'median', 'std'])
print(gender_income.round(0))
male_avg = df[df['Gender']=='Male']['MonthlyIncome'].mean()
female_avg = df[df['Gender']=='Female']['MonthlyIncome'].mean()
gap = ((male_avg - female_avg) / female_avg) * 100
print(f'\nBrecha: hombres ganan {gap:.1f}% más en promedio.')

# 2. Attrition por género
print('\n## 2. Tasa de rotación por género')
attrition_gender = pd.crosstab(df['Gender'], df['Attrition'], normalize='index')
print(attrition_gender.round(3))

# 3. Features con potencial sesgado
print('\n## 3. Features que podrían codificar sesgos')
print('- Gender: variable protegida. ¿Se usa directamente? SÍ (error ético).')
print('- Department: puede correlacionar con género (HR vs Sales).')
print('- MonthlyIncome: resultado de la brecha salarial, no causa.')
print('- Education: puede ser proxy de estatus socioeconómico.')

# 4. Recomendación
print('\n## 4. Recomendaciones éticas')
print('1. EXCLUIR Gender del modelo (variable protegida).')
print('2. Evaluar si MonthlyIncome es causa o consecuencia de la rotación.')
print('3. Auditar predicciones por subgrupo (¿predice peor para mujeres?).')
print('4. Documentar limitaciones y sesgos conocidos.')
print('5. Incluir a RRHH en la revisión del modelo (no solo data science).')

In [ ]:
# Modelo sin Gender para comparar
print('\n## 5. Entrenar modelo SIN Gender')

X_no_gender = X.drop('Gender', axis=1)
X_train_ng, X_test_ng, y_train_ng, y_test_ng = train_test_split(
    X_no_gender, y, test_size=0.20, random_state=42, stratify=y
)

rf_ng = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_split=10,
    min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_ng.fit(X_train_ng, y_train_ng)

y_pred_ng = rf_ng.predict(X_test_ng)
y_proba_ng = rf_ng.predict_proba(X_test_ng)[:, 1]

print(f'\nCon Gender:     F1={f1_score(y_test, y_pred_rf):.4f}, AUC={roc_auc_score(y_test, y_proba_rf):.4f}')
print(f'Sin Gender:     F1={f1_score(y_test_ng, y_pred_ng):.4f}, AUC={roc_auc_score(y_test_ng, y_proba_ng):.4f}')
print(f'\nConclusión: ¿Se pierde mucho rendimiento sin Gender?')
print('Si la pérdida es pequeña, es correcto eliminarlo por razones éticas.')
print('Si la pérdida es grande, investigar por qué (¿Gender es proxy de algo?).')